In [1]:
import glob
import pandas as pd
from helper import *
from torch.optim import Adam
from torchinfo import summary
from torch.utils.data import DataLoader

I moved classes and functions from the previous notebook to the `helper.py` not to rewrite the same code

In [2]:
HR_train_paths = sorted(glob.glob("../data/DIV2K_train_HR/*.png"))
X2_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X2/*.png"))
X4_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X4/*.png"))
X8_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X8/*.png"))
X16_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X16/*.png"))
X32_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X32/*.png"))
X64_train_paths = sorted(glob.glob("../data/DIV2K_train_LR_bicubic/X64/*.png"))

HR_valid_paths = sorted(glob.glob("../data/DIV2K_valid_HR/*.png"))
X2_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X2/*.png"))
X4_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X4/*.png"))
X8_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X8/*.png"))
X16_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X16/*.png"))
X32_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X32/*.png"))
X64_valid_paths = sorted(glob.glob("../data/DIV2K_valid_LR_bicubic/X64/*.png"))

## X8 Scaling

The process of preparation for training is the same as in 4X notebook, but the starting point is the **4X model**, not the 2X model

In [3]:
# checkpoint = torch.load('./model_checkpoints/SRResNet/X4.pth')
model = SRResNet(4).to(device)
# model.load_state_dict(checkpoint['model_state_dict'])

model.upscaling_head = model.upscaling_head[:-1]

head = nn.Sequential(
    nn.Conv2d(64, 256, 3, stride=1, padding='same'),
    nn.PixelShuffle(2),
    nn.PReLU(),
    nn.Conv2d(64, 3, 9, stride=1, padding='same')
)

model.upscaling_head = nn.Sequential(*model.upscaling_head, *head)

for param in model.expand.parameters():
    param.requires_grad = False

for param in model.residual_blocks.parameters():
    param.requires_grad = False

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = False

summary(model, col_names=['trainable'])

Layer (type:depth-idx)                   Trainable
EDSR                                     Partial
├─Sequential: 1-1                        False
│    └─Conv2d: 2-1                       False
│    └─PReLU: 2-2                        False
├─Sequential: 1-2                        False
│    └─ResBlockEDSR: 2-3                 False
│    │    └─Sequential: 3-1              False
│    └─ResBlockEDSR: 2-4                 False
│    │    └─Sequential: 3-2              False
│    └─ResBlockEDSR: 2-5                 False
│    │    └─Sequential: 3-3              False
│    └─ResBlockEDSR: 2-6                 False
│    │    └─Sequential: 3-4              False
│    └─ResBlockEDSR: 2-7                 False
│    │    └─Sequential: 3-5              False
│    └─ResBlockEDSR: 2-8                 False
│    │    └─Sequential: 3-6              False
│    └─ResBlockEDSR: 2-9                 False
│    │    └─Sequential: 3-7              False
│    └─ResBlockEDSR: 2-10                False
│    │ 

In [ ]:
valid_ds = SRResNet_Dataset(HR_valid_paths, 8, ram_limit_gb=1)

In [ ]:
train_ds = SRResNet_Dataset(HR_train_paths, 8, ram_limit_gb=8)

In [ ]:
train_dl = DataLoader(train_ds, batch_size=16, shuffle=True, num_workers=os.cpu_count()-1)
valid_dl = DataLoader(valid_ds, batch_size=16, shuffle=False, num_workers=os.cpu_count()-1)

loss_fn = nn.L1Loss()
optimizer = Adam(model.upscaling_head[-4:].parameters(), lr=1e-4)
model = torch.compile(model)

train(model, train_dl, valid_dl, optimizer, loss_fn, 1000)

In [ ]:
for param in model.expand.parameters():
    param.requires_grad = True

for param in model.residual_blocks.parameters():
    param.requires_grad = True

for param in model.upscaling_head[:-4].parameters():
    param.requires_grad = True

In [ ]:
optimizer = Adam(model.parameters(), lr=1e-5)

train(model, train_dl, valid_dl, optimizer, loss_fn, 5000)

In [ ]:
checkpoint = torch.load('../model_checkpoints/SRResNet/X8.pth')
model = SRResNet(8).to(device)
model.load_state_dict(checkpoint['model_state_dict'])

In [ ]:
model.eval();

In [ ]:
targets = [X8_valid_paths, X4_valid_paths, X2_valid_paths, HR_valid_paths]

metrics_x8 = pd.DataFrame(columns=["PSNR↑", "SSIM↑", "LPIPS↓"])
for target_ds in tqdm(targets, total=4):
    metrics_x8.loc[len(metrics_x8)] = calc_metrics(model, target_ds, 8)

metrics_x8.index = ["31px -> 248px", "63px -> 504px", "127px -> 1016px", "255px -> 2040px"]
metrics_x8

### Super-resolution showcase

In [ ]:
print("31px -> 248px")
inp = transform(Image.open(X64_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/31px', exist_ok=True)
img.save('../image_results/8X/31px/SRResNet_31px.png')
img

In [ ]:
print("63px -> 504px")
inp = transform(Image.open(X32_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/63px', exist_ok=True)
img.save('../image_results/8X/63px/SRResNet_63px.png')
img

In [ ]:
print("127px -> 1016px")
inp = transform(Image.open(X16_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/127px', exist_ok=True)
img.save('../image_results/8X/127px/SRResNet_127px.png')
img

In [ ]:
print("255px -> 2040px")
inp = transform(Image.open(X8_valid_paths[4])).to(device)
with torch.inference_mode():
    out = model(inp).clamp(0.0, 1.0) * 255.0

img = Image.fromarray(out.permute(1, 2, 0).to(torch.uint8).cpu().numpy())
os.makedirs('../image_results/8X/255px', exist_ok=True)
img.save('../image_results/8X/255px/SRResNet_255px.png')
img